In [16]:
import polars as pl
import matplotlib.pyplot as plt

In [17]:
df = pl.scan_csv("../data/analysis/india.csv")

In [18]:
# Find indicators present consistently across 2000-2023
# (ignore 2024-2025, they're incomplete)
filtered = (
    df.filter(pl.col("REF_AREA") == "IND")
    .filter(pl.col("SEX") == "_T")
    .filter(pl.col("AGE") == "_T")
    .filter(pl.col("URBANISATION") == "_T")
    .filter(pl.col("OBS_VALUE").is_not_null())
    .filter(pl.col("TIME_PERIOD").cast(pl.Int32).is_between(2000, 2023))
    .group_by("INDICATOR")
    .agg([
        pl.col("TIME_PERIOD").n_unique().alias("year_count"),
        pl.col("INDICATOR_LABEL").first()
    ])
    .filter(pl.col("year_count") >= 23)  # present in at least 20 of 24 years
    .sort("year_count", descending=True)
)
filtered.collect()

INDICATOR,year_count,INDICATOR_LABEL
str,u32,str
"""WB_WDI_IE_PPN_TRAN_CD""",24,"""Public private partnerships in…"
"""WB_WDI_BN_FIN_TOTL_CD""",24,"""Net financial account (BoP, cu…"
"""WB_WDI_SM_POP_NETM""",24,"""Net migration"""
"""WB_WDI_DC_DAC_KORL_CD""",24,"""Net bilateral aid flows from D…"
"""WB_WDI_DC_DAC_NLDL_CD""",24,"""Net bilateral aid flows from D…"
…,…,…
"""WB_WDI_FM_AST_PRVT_ZG_M3""",23,"""Claims on private sector (annu…"
"""WB_WDI_ER_H2O_FWTL_K3""",23,"""Annual freshwater withdrawals,…"
"""WB_WDI_TM_VAL_ICTG_ZS_UN""",23,"""ICT goods imports (% total goo…"


In [19]:
india_ind = (
    filtered.select(["INDICATOR", "year_count","INDICATOR_LABEL"])
    .unique()
    .sort("year_count")
    .collect()
)
with open("../data/analysis/india_indicator.md","w",encoding="utf-8") as f:
    f.write("# INDIA INDICATOR\n\n")
    f.write("| INDICATOR | year_count | INDICATOR_LABEL |\n")
    f.write("|------|-----------|-----------|\n")

    for row in india_ind.iter_rows(named=True):
        f.write(
            f"| {row['INDICATOR']}| {row['year_count']} | {row['INDICATOR_LABEL']}|\n"
        )

In [20]:
target = [
    "WB_WDI_NY_GDP_PCAP_KD", "WB_WDI_NY_GDP_MKTP_KD_ZG",
    "WB_WDI_FP_CPI_TOTL_ZG", "WB_WDI_NE_TRD_GNFS_ZS",
    "WB_WDI_SP_DYN_LE00_IN", "WB_WDI_SH_DYN_MORT",
    "WB_WDI_SH_STA_MMRT", "WB_WDI_SH_XPD_CHEX_GD_ZS",
    "WB_WDI_SE_SEC_ENRR", "WB_WDI_SE_TER_ENRR",
    "WB_WDI_IT_NET_USER_ZS", "WB_WDI_IT_CEL_SETS_P2",
    "WB_WDI_SP_POP_TOTL", "WB_WDI_SP_POP_GROW",
    "WB_WDI_SL_UEM_TOTL_ZS", "WB_WDI_SL_TLF_CACT_ZS",
    "WB_WDI_EN_GHG_ALL_MT_CE_AR5", "WB_WDI_EG_USE_PCAP_KG_OE",
    "WB_WDI_EG_ELC_ACCS_ZS", "WB_WDI_EG_USE_COMM_FO_ZS"
]

india_WD = (
    df.filter(pl.col("INDICATOR").is_in(target))
    .filter(pl.col("REF_AREA") == "IND")
    .filter(pl.col("SEX") == "_T")
    .filter(pl.col("AGE") == "_T")
    .filter(pl.col("URBANISATION") == "_T")
    .select(["INDICATOR", "INDICATOR_LABEL", "TIME_PERIOD", "OBS_VALUE"])
    .sort(["INDICATOR", "TIME_PERIOD"])
    .collect()
)

In [22]:
india_WD.head()

INDICATOR,INDICATOR_LABEL,TIME_PERIOD,OBS_VALUE
str,str,i64,f64
"""WB_WDI_EG_ELC_ACCS_ZS""","""Access to electricity (% of po…",1993,50.9
"""WB_WDI_EG_ELC_ACCS_ZS""","""Access to electricity (% of po…",1994,49.81131
"""WB_WDI_EG_ELC_ACCS_ZS""","""Access to electricity (% of po…",1995,51.408775
"""WB_WDI_EG_ELC_ACCS_ZS""","""Access to electricity (% of po…",1996,53.003521
"""WB_WDI_EG_ELC_ACCS_ZS""","""Access to electricity (% of po…",1997,54.594864
